# Exploration: AAPL next-day direction
Quick look at the engineered features and the class balance before modelling. The main pipeline lives in `src/`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import download_prices
from src.features import build_features, feature_columns

prices = download_prices('AAPL')
df = build_features(prices)
features = feature_columns(df)
df[features].describe().T.round(3)

## Class balance
If one class dominates, accuracy alone is misleading, so compare models against a majority-class baseline.

In [ ]:
df['target'].value_counts(normalize=True).rename({1: 'up', 0: 'down/flat'})

## Distribution of daily returns
Fat tails and roughly zero mean are why direction is hard to predict.

In [ ]:
sns.histplot(df['next_return'], bins=100, color='#2a78d6')
plt.title('AAPL next-day return distribution'); plt.xlabel('return'); plt.show()

## How much does each feature correlate with tomorrow's return?

In [ ]:
corr = df[features].corrwith(df['next_return']).sort_values()
corr.plot.barh(figsize=(7, 8), color='#2a78d6', title='Correlation with next-day return')
plt.show()

## RSI vs next-day return
A classic mean-reversion idea: oversold (RSI < 30) leads to bounces. Is there visible structure?

In [ ]:
df['rsi_bucket'] = pd.cut(df['rsi_14'], [0, 30, 40, 50, 60, 70, 100])
df.groupby('rsi_bucket', observed=True)['target'].agg(['mean', 'count']).rename(columns={'mean': 'P(up)'})